# BỐI CẢNH PHÂN TÍCH - GIAI ĐOẠN KHỞI NGHIỆP 


## I. TỪ BÀI TOÁN QUẢN TRỊ ĐẾN CÂU HỎI PHÂN TÍCH 




Khi một startup như Pizza Runner chập chững bước vào thị trường, nỗi ám ảnh lớn nhất của người đứng bếp – đồng thời cũng là CEO – chính là câu hỏi tưởng chừng đơn giản nhưng hệ trọng: **“Liệu quán có bán được hàng không?”** 

Danny không có phòng kinh doanh, không sở hữu hệ thống CRM phức tạp. Mọi quyết định ban đầu đều dựa vào trực giác và kinh nghiệm non trẻ. Thế nhưng, chính những ngày đầu ấy lại đặt ra hàng loạt câu hỏi sống còn: Khách hàng đầu tiên là ai? Họ gọi món gì? Đơn hàng đầu tiên đến vào lúc nào, và liệu sẽ có đơn thứ hai? 

Nếu không trả lời được những câu hỏi này, Danny sẽ không thể sắp xếp ca làm cho đầu bếp một cách hợp lý. Anh cũng chẳng biết nên dự trữ bao nhiêu bột, bao nhiêu phô mai – và tệ hơn, không có con số nào để biết mình đang tiến về phía trước, hay đang lãng phí thời gian chờ ngày phá sản.

Nhưng khó khăn cũng chính là cơ hội. Dữ liệu tuy nhỏ bé, song lại sạch sẽ và phản ánh chân thực từng giao dịch. Chính vì vậy, giai đoạn khởi nghiệp này không cần những phân tích cầu kỳ. Nó chỉ cần tập trung vào một nhiệm vụ duy nhất: **thiết lập đường cơ sở (baseline)** cho các chỉ số cốt lõi – nền móng cho mọi quyết định sau này. 

Bảng dưới đây tóm tắt những mục tiêu quản trị cấp thiết nhất, cùng các câu hỏi phân tích tương ứng, được kế thừa trực tiếp từ tầm nhìn chung trong file `00_Business_Context`.


| Câu hỏi quản trị | Câu hỏi phân tích |
|------------------|-------------------|
| Quy mô bán hàng có đủ để duy trì hoạt động? | – Tổng số pizza đã đặt (A.1)<br>– Số đơn hàng riêng biệt (A.2) |
| Sản phẩm nào đang kéo doanh thu chính? | – Số lượng mỗi loại pizza đã giao thành công (A.4)<br>– Số lượng Vegetarian và Meatlovers đặt bởi từng khách hàng (A.5) |
| Ai là khách hàng quan trọng nhất? | – Số lượng Vegetarian và Meatlovers đặt bởi từng khách hàng (A.5)<br>– Số pizza có thay đổi (exclusions/extras) theo từng khách (A.7) |
| Đơn hàng lớn nhất có thể là bao nhiêu? | – Số pizza tối đa trong một đơn hàng (chỉ tính đơn giao thành công) (A.6) |
| Khung giờ nào cần tăng cường nhân sự? | – Lưu lượng pizza theo giờ trong ngày (A.9)<br>– Lưu lượng đơn hàng theo ngày trong tuần (A.10) |
| Tỷ lệ hủy đơn có đáng báo động không? | – Số đơn giao thành công theo từng runner (A.3) – từ đó suy ra tỷ lệ hủy |

## II. PHÂN TÍCH VÀ ĐỀ XUẤT HÀNH ĐỘNG 

In [0]:
%sql
USE Pizza_Runner;

### NHÓM 1: ĐO LƯỜNG TỔNG CẦU VÀ XU HƯỚNG HÀNH ĐỘNG 

#### A.1. How many pizzas were ordered?

In [0]:
%sql
SELECT 
		COUNT(pizza_id) AS total_pizzas_ordered
FROM destination.customer_orders 

total_pizzas_ordered
14


#### A.2. How many unique customer orders were made?

In [0]:
%sql
SELECT 
	-- không cần UNIQUE vì mỗi dòng trong bảng orders là đại diện cho một đơn hàng được đặt 
	 COUNT(order_id) AS total_orders
FROM destination.orders

total_orders
10


#### A.9. What was the total volume of pizzas ordered for each hour of the day?

In [0]:
%sql
WITH RECURSIVE AllHours AS (
	SELECT 0 AS hour_of_day 
	UNION ALL 
	SELECT hour_of_day + 1 FROM AllHours WHERE hour_of_day < 23
)
SELECT
		ah.hour_of_day,
		COALESCE(COUNT(c.pizza_id), 0) AS total_pizzas
FROM Allhours ah
LEFT JOIN	destination.orders o ON ah.hour_of_day = HOUR(o.order_time)
LEFT JOIN   destination.customer_orders c ON o.order_id = c.order_id
GROUP BY ah.hour_of_day
ORDER BY ah.hour_of_day

hour_of_day,total_pizzas
0,0
1,0
2,0
3,0
4,0
5,0
6,0
7,0
8,0
9,0


#### A.10. What was the volume of orders for each day of the week?

In [0]:
%sql
SELECT   
    DATE_FORMAT(o.order_time, 'EEEE') AS day_of_week, 
    COUNT(1) AS total_orders 
FROM destination.orders o
GROUP BY 
    DATE_FORMAT(o.order_time, 'EEEE'), 
    DAYOFWEEK(o.order_time)  
ORDER BY 
    DAYOFWEEK(o.order_time) ASC; - 

day_of_week,total_orders
Wednesday,5
Thursday,2
Friday,1
Saturday,2


### NHÓM 2: CƠ CẤU SẢN PHẨM VÀ SỞ THÍCH KHÁCH HÀNG

#### A.4. How many of each type of pizza was delivered?

In [0]:
%sql
SELECT 
		c.pizza_id,
		COUNT(customer_orders_id) AS delivered_pizzas
FROM destination.runner_orders r
INNER JOIN destination.orders o ON r.order_id = o.order_id
INNER JOIN destination.customer_orders c ON r.order_id = c.order_id 
WHERE r.cancellation IS NULL
GROUP BY c.pizza_id

pizza_id,delivered_pizzas
1,9
2,3


#### A.5. How many Vegetarian and Meatlovers were ordered by each customer?

In [0]:
%sql
SELECT 
		 o.customer_id,
		 p.pizza_name,
		 COUNT(c.customer_orders_id) AS ordered_pizzas
FROM destination.customer_orders c 
INNER JOIN destination.pizza_names p ON c.pizza_id = p.pizza_id
INNER JOIN destination.orders o ON c.order_id = o.order_id
GROUP BY  o.customer_id,
		  p.pizza_name
ORDER BY o.customer_id ASC

customer_id,pizza_name,ordered_pizzas
101,Vegetarian,1
101,Meatlovers,2
102,Meatlovers,2
102,Vegetarian,1
103,Meatlovers,3
103,Vegetarian,1
104,Meatlovers,3
105,Vegetarian,1


### NHÓM 3: MỨC ĐỘ TÙY CHỈNH VÀ HÀNH VI CÁ NHÂN 

#### A.7. For each customer, how many delivered pizzas had at least 1 change and how many had no changes?

In [0]:
%sql
-- CÁCH 1
WITH DeliveredPizzas AS (
		SELECT  
				c.customer_orders_id,
				o.customer_id,
				c.order_id,
				c.pizza_id,
				ex.topping_id AS extras,
				ec.topping_id AS exclusions
		FROM destination.customer_orders c
		LEFT JOIN destination.orders o ON c.order_id = o.order_id
		LEFT JOIN destination.runner_orders r ON o.order_id = r.order_id
		LEFT JOIN destination.customer_orders_extras ex ON c.customer_orders_id = ex.customer_orders_id
		LEFT JOIN destination.customer_orders_exclusions ec ON c.customer_orders_id = ec.customer_orders_id
		WHERE r.cancellation IS NULL -- đây là điều kiện để lấy đơn hàng đã giao, không lấy những đơn hàng đã đặt nhưng chưa giao do bị hủy 
)
, Change_Or_Not_Changes AS (
SELECT 
	 d.customer_orders_id ,
	 d.customer_id,
	 d.order_id,
	 d.pizza_id,
	 -- Sử dụng MAX để kiểm tra: chỉ cần có 1 dòng extras/exclusions là cả pizza đó được tính là 'có thay đổi'
	 MAX(
	 CASE 
		WHEN (d.extras IS NOT NULL) OR (d.exclusions IS NOT NULL) THEN 1
		ELSE 0
	 END) AS has_change
FROM DeliveredPizzas d
GROUP BY d.customer_orders_id,
		 d.customer_id,
		 d.order_id,
		 d.pizza_id
)
SELECT
	     c.customer_id,
		 SUM(has_change) AS count_pizza_with_changes,
		 SUM(CASE WHEN has_change = 0 THEN 1 ELSE 0 END) AS count_pizza_no_changes
FROM Change_Or_Not_Changes c
GROUP BY c.customer_id
ORDER BY c.customer_id

customer_id,count_pizza_with_changes,count_pizza_no_changes
101,0,2
102,0,3
103,3,0
104,2,1
105,1,0


In [0]:
%sql
-- CÁCH 2: CÓ TỒN TẠI HAY KHÔNG -> CASE WHEN + EXISTS 
WITH Change_pizzas AS(
SELECT 
	co.customer_orders_id,
	o.customer_id,
	o.order_id,
	co.pizza_id,
	-- Lấy customer_orders_id ở extras
	CASE
		WHEN EXISTS(SELECT ex.customer_orders_id FROM destination.customer_orders_extras ex WHERE ex.customer_orders_id = co.customer_orders_id) 
	-- Lấy customer_orders_id ở exclusions 
			OR EXISTS(SELECT ec.customer_orders_id FROM destination.customer_orders_exclusions ec WHERE ec.customer_orders_id = co.customer_orders_id) THEN 1
		ELSE 0
	END AS has_changed
FROM destination.customer_orders co
JOIN destination.orders o ON co.order_id = o.order_id
JOIN destination.runner_orders ro ON o.order_id = ro.order_id
WHERE ro.cancellation IS NULL
)
SELECT
		c.customer_id,
		SUM(c.has_changed) AS changed_pizzas,
		SUM(CASE WHEN c.has_changed = 0 THEN 1 ELSE 0 END) AS no_change
FROM Change_pizzas c
GROUP BY c.customer_id 
ORDER BY c.customer_id ASC

customer_id,changed_pizzas,no_change
101,0,2
102,0,3
103,3,0
104,2,1
105,1,0


#### A.8. How many pizzas were delivered that had both exclusions and extras?

In [0]:
%sql
-- Câu này sẽ dùng 1 phần câu 7 

-- CÁCH 1
-- nhưng đổi 1 chút ở case when 

WITH DeliveredPizzas AS (
		SELECT  
				c.customer_orders_id,
				o.customer_id,
				c.order_id,
				c.pizza_id,
				ex.topping_id AS extras,
				ec.topping_id AS exclusions
		FROM destination.customer_orders c
		LEFT JOIN destination.orders o ON c.order_id = o.order_id
		LEFT JOIN destination.runner_orders r ON o.order_id = r.order_id
		LEFT JOIN destination.customer_orders_extras ex ON c.customer_orders_id = ex.customer_orders_id
		LEFT JOIN destination.customer_orders_exclusions ec ON c.customer_orders_id = ec.customer_orders_id
		WHERE r.cancellation IS NULL -- đây là điều kiện để lấy đơn hàng đã giao, không lấy những đơn hàng đã đặt nhưng chưa giao do bị hủy 
)
, Change_Both AS (
SELECT 
	 d.customer_orders_id ,
	 d.customer_id,
	 d.order_id,
	 d.pizza_id,
	 -- Sử dụng MAX để kiểm tra: chỉ cần có 1 dòng extras/exclusions là cả pizza đó được tính là 'có thay đổi'
	 MAX(CASE 
		WHEN (d.extras IS NOT NULL) AND (d.exclusions IS NOT NULL) THEN 1
		ELSE 0
	 END) AS has_change_both 
FROM DeliveredPizzas d
GROUP BY d.customer_orders_id,
		 d.customer_id,
		 d.order_id,
		 d.pizza_id
)
SELECT 
		COUNT(b.customer_orders_id) AS count_pizzas_that_has_both_exclu_extras
FROM Change_Both b
WHERE b.has_change_both = 1

count_pizzas_that_has_both_exclu_extras
1


In [0]:
%sql
-- CÁCH 2: CÓ TỒN TẠI CẢ EXTRAS VÀ EXCLUSIONS HAY KHÔNG -> CASE WHEN + EXISTS 
WITH Change_pizzas AS(
		SELECT 
			co.customer_orders_id,
			o.customer_id,
			o.order_id,
			co.pizza_id,
			-- Lấy customer_orders_id ở extras
			CASE
				WHEN EXISTS(SELECT ex.customer_orders_id FROM destination.customer_orders_extras ex WHERE ex.customer_orders_id = co.customer_orders_id) 
			-- Lấy customer_orders_id ở exclusions 
					AND EXISTS(SELECT ec.customer_orders_id FROM destination.customer_orders_exclusions ec WHERE ec.customer_orders_id = co.customer_orders_id) THEN 1
				ELSE 0
			END AS has_both_changed
		FROM destination.customer_orders co
		JOIN destination.orders o ON co.order_id = o.order_id
		JOIN destination.runner_orders ro ON o.order_id = ro.order_id
		WHERE ro.cancellation IS NULL
)
SELECT
		c.customer_orders_id,
		-- HOW MANY -> DÙNG COUNT 
		COUNT(c.has_both_changed) AS has_both_changed
FROM Change_pizzas c
WHERE c.has_both_changed = 1
GROUP BY c.customer_orders_id

customer_orders_id,has_both_changed
14,1


In [0]:
%sql
-- CÁCH 3: 
SELECT 
    COUNT(co.customer_orders_id) AS count_pizzas_both_changed
FROM destination.customer_orders co
JOIN destination.orders o ON co.order_id = o.order_id
JOIN destination.runner_orders ro ON o.order_id = ro.order_id
WHERE ro.cancellation IS NULL -- Chỉ lấy đơn đã giao
  AND EXISTS (SELECT 1 FROM destination.customer_orders_extras ex 
              WHERE ex.customer_orders_id = co.customer_orders_id) -- Có Extras
  AND EXISTS (SELECT 1 FROM destination.customer_orders_exclusions ec 
              WHERE ec.customer_orders_id = co.customer_orders_id); -- Và có Exclusions

count_pizzas_both_changed
1


### NHÓM 4: NĂNG LỰC PHỤC VỤ VÀ CHẤT LƯỢNG GIAO HÀNG 

#### A.6. What was the maximum number of pizzas delivered in a single order?

In [0]:
%sql
-- CÁCH 1
WITH DeliveredOrders AS (
	-- Bước 1: Chỉ lấy các đơn hàng đã giao thành công và đếm số lượng pizza
	SELECT 
			c.order_id,
			COUNT(c.pizza_id) AS pizza_count 
	FROM destination.customer_orders c
	INNER JOIN destination.runner_orders r ON c.order_id = r.order_id
	WHERE r.cancellation IS NULL -- không lấy những đơn hàng bị hủy 
	GROUP BY c.order_id
),
RankedOrders AS(
	-- Bước 2: Xếp hạng các đơn hàng dựa trên số lượng pizza
	SELECT 
		  order_id,
		  pizza_count,
		  -- DÙNG DENSE_RANK vì cho phép đồng hạng 
		  DENSE_RANK() OVER(ORDER BY pizza_count DESC) AS rank_num
	FROM  DeliveredOrders
)
-- Bước 3: Chỉ lấy ra đơn hàng top 1
SELECT
		order_id,
		pizza_count AS max_pizzas_delivered 
FROM RankedOrders
WHERE rank_num = 1

order_id,max_pizzas_delivered
4,3


#### A.3. How many successful orders were delivered by each runner?

In [0]:
%sql
SELECT 
    runner_id,
    COUNT(order_id) AS successful_orders
FROM destination.runner_orders
WHERE cancellation IS NULL
GROUP BY runner_id;

runner_id,successful_orders
1,4
2,3
3,1


## III. TỔNG KẾT 

- Chỉ số / KPI
- Giá trị hiện tại
- Hành động đề xuất
- Phòng ban chịu trác nhiệm 

## PHỤ LỤC: KHÁC BIỆT CÚ PHÁP GIỮA T‑SQL VÀ DATABRICKS SQL 


Trong quá trình phân tích, một số câu lệnh yêu cầu điều chỉnh cú pháp khi chuyển đổi giữa hai môi trường. Bảng dưới đây tổng hợp các điểm khác biệt chính, kèm giải thích ngắn gọn về nguyên nhân.

| Vấn đề | T‑SQL (SQL Server) | Databricks SQL (Spark SQL) | Giải thích |
|--------|-------------------|----------------------------|------------|
| **Trích xuất giờ từ datetime** | `DATEPART(HOUR, order_time)` | `HOUR(order_time)` | Spark cung cấp các hàm nguyên tố như `HOUR`, `MINUTE`, `SECOND` – trực tiếp và gần với tư duy lập trình. <br> `DATEPART` là hàm độc quyền của T‑SQL, không có trong chuẩn chung. |
| **Thay thế giá trị NULL** | `ISNULL(column, 0)` | `COALESCE(column, 0)` hoặc `IFNULL(column, 0)` | `ISNULL` chỉ chấp nhận hai tham số và là đặc trưng của T‑SQL. <br> `COALESCE` theo chuẩn ANSI, có thể nhận nhiều tham số, trả về giá trị khác NULL đầu tiên – phù hợp với yêu cầu xử lý đa dạng trên dữ liệu lớn. |
| **Tên ngày trong tuần** | `DATENAME(WEEKDAY, order_time)` | `DATE_FORMAT(order_time, 'EEEE')` | T‑SQL dùng `DATENAME` và cho kết quả phụ thuộc ngôn ngữ cài đặt máy chủ. <br> Spark sử dụng mẫu định dạng Java (`EEEE` cho tên đầy đủ), cho phép kiểm soát độc lập và nhất quán, bất kể môi trường triển khai. |
| **Số thứ tự ngày trong tuần** | `DATEPART(WEEKDAY, order_time)` | `DAYOFWEEK(order_time)` (1=Chủ nhật, 7=Thứ Bảy) | Giá trị trả về của `DATEPART(WEEKDAY...)` thay đổi theo thiết lập `@@DATEFIRST` trên từng máy chủ SQL Server. <br> Spark cố định `DAYOFWEEK` với quy ước 1 là Chủ nhật, giúp kết quả thống nhất trên toàn cụm, tránh sai lệch khi tổng hợp dữ liệu từ nhiều nguồn. |
| **CTE đệ quy** | `WITH cte AS (SELECT ... UNION ALL SELECT ... FROM cte WHERE ...)` | `WITH RECURSIVE cte AS (SELECT ... UNION ALL SELECT ... FROM cte WHERE ...)` | Chuẩn ANSI SQL yêu cầu từ khóa `RECURSIVE` để phân biệt rõ ràng CTE đệ quy. <br> SQL Server cho phép bỏ qua, nhưng Spark tuân thủ nghiêm ngặt để tránh nhập nhằng khi phân tích kế hoạch thực thi trên môi trường phân tán. |
| **Lấy các dòng đồng hạng cao nhất** | `SELECT TOP 1 WITH TIES ... ORDER BY cnt DESC` | Dùng hàm cửa sổ: `SELECT ... FROM (SELECT *, DENSE_RANK() OVER (ORDER BY cnt DESC) AS rnk FROM ...) WHERE rnk = 1` | `TOP WITH TIES` là cú pháp viết tắt của T‑SQL, không có trong chuẩn SQL. <br> Spark yêu cầu cách tiếp cận tường minh bằng `DENSE_RANK()` – linh hoạt hơn và hoạt động ổn định trên các tập dữ liệu được phân vùng, xử lý song song. |